Salmon Notebook:
https://colab.research.google.com/drive/1smWcPESgD4eVZ2BiodCBPJLSArCVWtyc#scrollTo=AkTRuFR7sirF

In [ ]:
print("I am learning AI")

I am learning AI


In [ ]:
def is_prime(n):
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True

# Test cases
test_numbers = [2, 4, 17, 20, 97, 100]
for num in test_numbers:
    print(f"Is {num} prime? {is_prime(num)}")

Is 2 prime? True
Is 4 prime? False
Is 17 prime? True
Is 20 prime? False
Is 97 prime? True
Is 100 prime? False


# config.json

{
  "OLLAMA_API_KEY":"<yourkey>"
}

In [6]:
import json
import requests

# Load the API key from the configuration file
try:
    with open('/content/config.json', 'r') as f:
        config = json.load(f)
    api_key = config.get('OLLAMA_API_KEY')
    if not api_key:
        raise ValueError("API key not found in config.json under 'OLLAMA_API_KEY'")
except FileNotFoundError:
    print("Error: /content/config.json file not found.")
    api_key = None
except Exception as e:
    print(f"Error reading config: {e}")
    api_key = None

if api_key:
    # Ask the user for a question
    user_question = input("Enter your question for the AI: ")

    # Ollama Cloud / API request setup using API chat endpoint
    api_url = "https://ollama.com/api/chat"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "gpt-oss:20b-cloud",
        "messages": [
            {"role": "user", "content": user_question}
        ],
        "stream": False
    }

    try:
        response = requests.post(api_url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()

        # Print the response content (supporting both OpenAI format and standard Ollama response keys)
        # Ollama API format returns { "message": { "role": "assistant", "content": "..." } }
        if 'choices' in result:
            answer = result['choices'][0]['message']['content']
        elif 'message' in result:
            answer = result['message']['content']
        else:
            answer = str(result)

        print("\n--- AI Response ---")
        print(answer)
    except Exception as e:
        print(f"An error occurred while making the request: {e}")

Enter your question for the AI: What is an AI agent? tell me in 3 sentences

--- AI Response ---
An AI agent is a software component that perceives its environment through sensors or data inputs, processes that information using algorithms or machine learning models, and takes actions to fulfill its objectives. It can operate autonomously, learning and adapting over time, or it may be guided by human instructions. Examples range from simple rule‑based chatbots to complex autonomous vehicles that navigate real‑world environments.


# Sequential Ollama API requests

In [8]:
import json
import requests
import time

# Load the API key from the configuration file
try:
    with open('/content/config.json', 'r') as f:
        config = json.load(f)
    api_key = config.get('OLLAMA_API_KEY')
    if not api_key:
        raise ValueError("API key not found in config.json under 'OLLAMA_API_KEY'")
except FileNotFoundError:
    print("Error: /content/config.json file not found.")
    api_key = None
except Exception as e:
    print(f"Error reading config: {e}")
    api_key = None

if api_key:
    # Ask the user for a question
    user_question = input("Enter your question for the AI: ")

    # Ollama Cloud / API request setup using API chat endpoint
    api_url = "https://ollama.com/api/chat"
    # so that ollama knows who is sending the req
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # Defined targets for the three separate requests
    word_targets = [50, 100, 250]

    # Track the start time of all requests
    start_time = time.time()

    for target in word_targets:
        # Formulate instructions forcing the response length restriction
        prompt_with_constraint = f"{user_question}\n\nCRITICAL INSTRUCTION: Respond in exactly or approximately {target} words."

        payload = {
            "model": "gpt-oss:20b-cloud",
            "messages": [
                {"role": "user", "content": prompt_with_constraint}
            ],
            "stream": False # don't show answer word by word, give full ans
        }

        try:
            response = requests.post(api_url, headers=headers, json=payload)
            response.raise_for_status()
            result = response.json()

            # Parse the response
            if 'choices' in result:
                answer = result['choices'][0]['message']['content']
            elif 'message' in result:
                answer = result['message']['content']
            else:
                answer = str(result)

            print(f"\n--- AI Response ({target} words target) ---")
            print(answer)
            print(f"Actual word count: {len(answer.split())} words.")
        except Exception as e:
            print(f"An error occurred while making the request for {target} words: {e}")

    # Calculate and print total time taken
    end_time = time.time()
    total_duration = end_time - start_time
    print(f"\n========================================")
    print(f"Total time taken for all requests: {total_duration:.2f} seconds")
    print(f"========================================")

Enter your question for the AI: what is ollama

--- AI Response (50 words target) ---
Ollama is a lightweight, open‑source platform that lets developers run large language models locally for on‑device AI. It supports Llama‑based models from Meta and other architectures, offering a simple CLI, REST API, and Python bindings for rapid prototyping and private inference without cloud dependencies, and ease of deployment today.
Actual word count: 49 words.

--- AI Response (100 words target) ---
Ollama is a lightweight framework that lets developers run large language models on consumer hardware. It packages models in Docker‑style containers and exposes a REST API, so you can spin up GPT‑like inference locally without cloud access. The project ships a curated set of models such as Llama‑2, Mistral, and OpenChat, each built as an “Ollama image.” Users interact through a command‑line interface or the provided HTTP endpoints, and the API accepts text prompts and returns streamed completions. Ol

# Parallel Ollama API requests

Depends on your pricing tier, how many requests can be processed parellely
Free Ollama - no parallel reqs

In [9]:
# Identifiable Code: Parallel Ollama API requests with real-time response printing and total duration tracking
import json
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# Load the API key from the configuration file
try:
    with open('/content/config.json', 'r') as f:
        config = json.load(f)
    api_key = config.get('OLLAMA_API_KEY')
    if not api_key:
        raise ValueError("API key not found in config.json under 'OLLAMA_API_KEY'")
except FileNotFoundError:
    print("Error: /content/config.json file not found.")
    api_key = None
except Exception as e:
    print(f"Error reading config: {e}")
    api_key = None

if api_key:
    # Ask the user for a question
    user_question = input("Enter your question for the AI: ")

    # Ollama Cloud / API request setup using API chat endpoint
    api_url = "https://ollama.com/api/chat"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # Defined targets for the three separate requests
    word_targets = [50, 100, 250]

    def send_single_request(target):
        # Formulate instructions forcing the response length restriction
        prompt_with_constraint = f"{user_question}\n\nCRITICAL INSTRUCTION: Respond in exactly or approximately {target} words."

        payload = {
            "model": "gpt-oss:20b-cloud",
            "messages": [
                {"role": "user", "content": prompt_with_constraint}
            ],
            "stream": False  # don't show answer word by word, give full ans
        }

        try:
            response = requests.post(api_url, headers=headers, json=payload)
            response.raise_for_status()
            result = response.json()

            # Parse the response
            if 'choices' in result:
                answer = result['choices'][0]['message']['content']
            elif 'message' in result:
                answer = result['message']['content']
            else:
                answer = str(result)
            return target, answer, None
        except Exception as e:
            return target, None, e

    # Track the start time of all parallel requests
    start_time = time.time()

    # Run requests concurrently and print responses immediately as they complete
    with ThreadPoolExecutor(max_workers=len(word_targets)) as executor:
        future_to_target = {executor.submit(send_single_request, target): target for target in word_targets}

        for future in as_completed(future_to_target):
            target, answer, error = future.result()
            if error:
                print(f"\nAn error occurred while making the request for {target} words: {error}")
            else:
                print(f"\n--- AI Response ({target} words target) ---")
                print(answer)
                print(f"Actual word count: {len(answer.split())} words.")

    # Calculate and print total time taken
    end_time = time.time()
    total_duration = end_time - start_time
    print(f"\n========================================")
    print(f"Total time taken for all parallel requests: {total_duration:.2f} seconds")
    print(f"========================================")

Enter your question for the AI: what is ollama

--- AI Response (100 words target) ---
Ollama is an open‑source platform that lets developers run large language models (LLMs) locally without relying on cloud services. Built by the team behind the popular Ollama AI app, the tool downloads models such as Llama‑2, Claude, or GPT‑Neo in lightweight formats and serves them through a local API. This approach reduces latency, keeps data private, and cuts costs associated with external inference endpoints. Ollama supports a command‑line interface and can be integrated into custom applications, making it a handy choice for prototyping, data‑science workflows, or building privacy‑centric AI solutions. Its user‑friendly design and robust model management make it popular among hobbyists and professionals alike.
Actual word count: 105 words.

--- AI Response (50 words target) ---
Ollama is an open‑source framework that lets developers build, fine‑tune, and run large language models locally or in th

Ollama also slows downs the requests if you send too many requests